# CNN Models

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, ConcatDataset, random_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm
import random
from collections import Counter

# === Configuration ===
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 20 
EARLY_STOPPING_PATIENCE = 2
LEARNING_RATE = 0.001
SUBSET_FRACTION = 1  # use 30% of data for faster training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === Dataset Paths ===
DATASET_PATHS = [
    "/kaggle/input/coffee-leaf-disease-dataset/dataset/Train",
    "/kaggle/input/coffee-leaf-disease-dataset/dataset/test",
    "/kaggle/input/coffee-leaf-diseases/train",
    "/kaggle/input/coffee-leaf-diseases/test",
    "/kaggle/input/ethiopian-coffee-leaf-disease/ethiopian cofee leaf dataset/train aug",
    "/kaggle/input/ethiopian-coffee-leaf-disease/ethiopian cofee leaf dataset/test",
    "/kaggle/input/jmuben-coffee-dataset/JMuBEN"
]



# === Transforms ===
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# === Load All Datasets ===
def load_datasets(paths):
    datasets_list = []
    for path in paths:
        if os.path.exists(path):
            ds = datasets.ImageFolder(path, transform=transform)
            datasets_list.append(ds)
    return ConcatDataset(datasets_list)

# === Class Distribution Viewer ===
def print_class_distribution(dataset):
    labels = [label for _, label in dataset]
    counter = Counter(labels)
    print("Class Distribution:")
    for i, count in counter.items():
        print(f"{dataset.classes[i]}: {count}")

# === Sample Image Viewer ===
def show_sample_images(dataset, name):
    fig, axes = plt.subplots(1, 3, figsize=(10, 4))
    indices = random.sample(range(len(dataset)), 3)
    for ax, idx in zip(axes, indices):
        img, label = dataset[idx]
        img = img.permute(1, 2, 0).numpy()
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        ax.set_title(dataset.classes[label])
        ax.axis("off")
    plt.suptitle(f"Sample Images from {name}")
    plt.tight_layout()
    plt.show()

# === Load and Display EDA ===
combined_dataset = load_datasets(DATASET_PATHS)

for dpath in DATASET_PATHS:
    dname = dpath.split("/")[-1]
    ds = datasets.ImageFolder(dpath, transform=transform)
    print_class_distribution(ds)
    show_sample_images(ds, dname)

# === Define Class Names ===
class_names = sorted(list(set([cls for dset in combined_dataset.datasets for cls in dset.classes])))

# === Use a Subset for Fast Training ===
subset_size = int(len(combined_dataset) * SUBSET_FRACTION)
combined_dataset, _ = random_split(combined_dataset, [subset_size, len(combined_dataset) - subset_size])

# === Split Dataset ===
train_size = int(0.7 * len(combined_dataset))
val_size = len(combined_dataset) - train_size
print(f"val_size {val_size} train_size {train_size}")
train_data, val_data = random_split(combined_dataset, [train_size, val_size])

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)

# === CNN Training + Evaluation Function with Early Stopping ===
def train_and_evaluate_model(model, model_name, train_loader, val_loader):
    print(f"\n🔧 Training: {model_name}")
    num_classes = len(class_names)

    if isinstance(model, models.MobileNetV3):
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
    elif isinstance(model, models.ShuffleNetV2):
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif isinstance(model, models.EfficientNet):
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    best_loss = float('inf')
    patience_counter = 0
    history = {"loss": [], "acc": []}

    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for x, y in tqdm(train_loader, desc=f"{model_name} Epoch {epoch+1}"):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            correct += (outputs.argmax(1) == y).sum().item()
            total += y.size(0)
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = correct / total
        history["loss"].append(epoch_loss)
        history["acc"].append(epoch_acc)
        print(f"Epoch {epoch+1} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}")
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            patience_counter = 0
            torch.save(model.state_dict(), f"{model_name}_best.pth")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print("Early stopping.")
                break

    model.load_state_dict(torch.load(f"{model_name}_best.pth"))
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            preds = out.argmax(1)
            y_true += y.cpu().tolist()
            y_pred += preds.cpu().tolist()

    acc = accuracy_score(y_true, y_pred) * 100
    prec = precision_score(y_true, y_pred, average='weighted') * 100
    rec = recall_score(y_true, y_pred, average='weighted') * 100
    f1 = f1_score(y_true, y_pred, average='weighted') * 100

    print(f"\n {model_name} Results:\nAccuracy: {acc:.2f}% | Precision: {prec:.2f}% | Recall: {rec:.2f}% | F1: {f1:.2f}%")
    print("\nClassification Report:\n", classification_report(y_true, y_pred))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"{model_name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    return {"model": model_name, "acc": acc, "prec": prec, "rec": rec, "f1": f1, "history": history}

# === Train and Evaluate All CNN Models ===
results = {}
models_dict = {
    #"ShuffleNet": models.shufflenet_v2_x1_0(weights=models.ShuffleNet_V2_X1_0_Weights.DEFAULT),
    "MobileNetV3": models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    #"EfficientNet": models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT),
}

for name, model in models_dict.items():
    results[name] = train_and_evaluate_model(model, name, train_loader, val_loader)

# === Comparison Plot ===
metrics = ["acc", "prec", "rec", "f1"]
for metric in metrics:
    plt.figure(figsize=(6, 4))
    sns.barplot(x=list(results.keys()), y=[results[m][metric] for m in results])
    plt.ylabel(f"{metric.upper()} (%)")
    plt.title(f"Model Comparison - {metric.upper()}")
    plt.ylim(0, 100)
    plt.show()

# === Training Curve Plot ===
for name in results:
    plt.figure(figsize=(10, 4))
    plt.plot(results[name]['history']['loss'], label="Loss")
    plt.plot(results[name]['history']['acc'], label="Accuracy")
    plt.title(f"Training Curve for {name}")
    plt.xlabel("Epoch")
    plt.legend()
    plt.show()